# Laya multi-question fine-tune

Trains a questions checkpoint, the one to use with jobhunter's `laya` scorer: three questions (stack role,
dealbreaker, fit) combined by gates and fitted cut-offs. (`train_alert.ipynb` trains the older single-question
alert checkpoint.) This notebook only calls functions in `training/laya_train.py`; the logic is unit-tested on a
laptop, this notebook runs it on a GPU and exports the result.

**When prompted, select the six files from the upload folder:** `laya_train.py` (from `training/`), `posting.py`
(from `jobhunter/`), `scoring_resume.txt` (your resume as plain text), `pool_old.jsonl` and `pool_jobhunter.jsonl`
(two pools of your own labelled jobs, for example jobs you labelled before and jobs jobhunter found; JSON lines
with `job_id`, `title`, `company`, `description`, `label` (notify, log or skip) and optionally `location`; this
method also needs `match_score` (1-10), `stack_role` and `dealbreaker` (text or null) on every row), and
`heldout_ids.txt` (job ids kept out of training, separated by spaces or line breaks, for example a benchmark set of
your own). `jobhunter export-feedback` writes a record of your verdicts, not these pools: turning verdicts into
pools is a manual step today.

**Runtime: A100** (Runtime > Change runtime type > A100 GPU) before running the cells below.


In [ ]:
!pip -q install laya==0.3.20 scikit-learn pandas scipy
from google.colab import drive, files
drive.mount('/content/drive')
import sys, time, hashlib, zipfile, os
from datetime import datetime, timezone
from pathlib import Path
import torch
UP = Path('/content/upload'); UP.mkdir(exist_ok=True)
print('Upload: laya_train.py, posting.py, scoring_resume.txt, pool_old.jsonl, pool_jobhunter.jsonl, heldout_ids.txt')
for name, data in files.upload().items():
    (UP / name).write_bytes(data)
EXPECTED_FILES = ('laya_train.py', 'posting.py', 'scoring_resume.txt', 'pool_old.jsonl', 'pool_jobhunter.jsonl',
                  'heldout_ids.txt')
missing = [name for name in EXPECTED_FILES if not (UP / name).exists()]
if missing:
    raise RuntimeError(f'Missing uploaded file(s): {missing}. Re-run this cell and select all six files.')
sys.path.insert(0, str(UP))
import laya_train as lt

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Go to Runtime > Change runtime type > A100 GPU, then rerun this cell.')
device = torch.device('cuda')
memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
settings = lt.Settings(micro_batch=4, grad_accum=16) if memory_gb < 70 else lt.Settings()
print(f'GPU memory {memory_gb:.0f} GB: micro_batch={settings.micro_batch}, grad_accum={settings.grad_accum} '
      '(effective batch 64)')

SAFE = Path('/content/drive/MyDrive/laya_questions/inprogress')


def questions_fingerprint(splits: dict) -> str:
    """sha256 over the sorted train job_ids, then '|', then the sorted select job_ids: identifies the exact data
    split a saved checkpoint was trained against, so a restore can refuse a checkpoint from a different run."""
    train_ids = ','.join(sorted(r['job_id'] for r in splits['train']))
    select_ids = ','.join(sorted(r['job_id'] for r in splits['select']))
    return hashlib.sha256(f'{train_ids}|{select_ids}'.encode('utf-8')).hexdigest()


def save_best_so_far(b):
    """Persist the best epoch (weights, history, a fingerprint of the data split and the base checkpoint note) to
    Drive as training goes, so a disconnect does not lose it."""
    SAFE.mkdir(parents=True, exist_ok=True)
    torch.save({'state': b.state, 'epoch': b.epoch, 'history': b.history,
               'fingerprint': questions_fingerprint(splits), 'base_note': base_note,
               'saved_at': datetime.now(timezone.utc).isoformat()},
              SAFE / 'best.pt')


In [ ]:
model, tok, base_cfg, base, base_note = lt.load_question_base(device)
print('Base checkpoint:', base_note)
held = set((UP / 'heldout_ids.txt').read_text().split())
rows = [r for r in lt.read_pool(UP / 'pool_old.jsonl', 'old', questions=True)
        + lt.read_pool(UP / 'pool_jobhunter.jsonl', 'jobhunter', questions=True) if r['job_id'] not in held]
splits = lt.split_rows(rows, settings)
by_question = lt.sample_questions(splits['train'], settings)
resume = (UP / 'scoring_resume.txt').read_text().strip()
train_items_by_q = {qid: lt.build_question_items(rs, resume, tok, settings, qid) for qid, rs in by_question.items()}
train_items = [it for its in train_items_by_q.values() for it in its]
select_items = {qid: lt.build_question_items(splits['select'], resume, tok, settings, qid) for qid in lt.QUESTIONS}
calibrate_items = {qid: lt.build_question_items(splits['calibrate'], resume, tok, settings, qid) for qid in lt.QUESTIONS}
health = {}
for split, rows_by_q, items_by_q in (('train', by_question, train_items_by_q),
                                     ('select', {q: splits['select'] for q in lt.QUESTIONS}, select_items),
                                     ('calibrate', {q: splits['calibrate'] for q in lt.QUESTIONS}, calibrate_items)):
    health[f'{split}, job ({settings.job_max_len})'] = lt.input_health(rows_by_q['stack_role'], items_by_q['stack_role'])
    health[f'{split}, resume + job ({settings.max_len})'] = lt.input_health(rows_by_q['fit'], items_by_q['fit'])
counts = lt.class_counts(train_items_by_q)
deal_counts = lt.dealbreaker_counts(rows)
from collections import Counter
for name, rs in splits.items():
    print(name, len(rs), dict(Counter(r['label'] for r in rs)))
print('Class counts after sampling:', counts)
print('Dealbreaker categories per pool:', deal_counts)
for name, h in health.items():
    print('Input health,', name, h)
print('Train items per question:', {qid: len(its) for qid, its in train_items_by_q.items()}, '- total', len(train_items))


In [ ]:
_epoch_t0 = time.time()


def _epoch_log(msg):
    global _epoch_t0
    now = time.time()
    print(f'{msg} ({(now - _epoch_t0) / 60:.1f} min)')
    _epoch_t0 = now


result = lt.train_questions(model, train_items, select_items, splits['select'], settings, device,
                            log=_epoch_log, on_best=save_best_so_far, pad_id=tok.pad_token_id)
model.load_state_dict(result.state)


**If Colab disconnected after at least one epoch:** rerun the setup and data cells, then this cell restores the best epoch from Drive if it matches this run's data split; continue from the calibration cell. When training finished in this session it does nothing.


In [ ]:
best_path = SAFE / 'best.pt'
if 'result' in globals():
    print('Training finished in this session: nothing to restore.')
elif best_path.exists():
    saved = torch.load(best_path, map_location=device)
    if saved.get('base_note') != base_note:
        raise RuntimeError(f"best.pt was trained from {saved.get('base_note')}, but this session loaded "
                           f"{base_note}; rerun the setup and data cells (the base download may have failed) "
                           "or retrain")
    if saved['fingerprint'] != questions_fingerprint(splits):
        raise RuntimeError(f"best.pt is from a different run (saved {saved['saved_at']}); retrain")
    model.load_state_dict(saved['state'])
    result = lt.TrainResult(saved['state'], saved['epoch'], None, saved['history'])
    print('Restored epoch', saved['epoch'], 'from', best_path, '- saved', saved['saved_at'])
else:
    print('No saved checkpoint found at', best_path, '- run the training cell above.')


In [ ]:
sel_logits = {qid: lt.predict_question_logits(model, its, settings, device, tok.pad_token_id)
              for qid, its in select_items.items()}
cal_logits = {qid: lt.predict_question_logits(model, its, settings, device, tok.pad_token_id)
              for qid, its in calibrate_items.items()}
report, temps, cutoffs = lt.build_questions_report(
    sel_logits, cal_logits, splits['select'], splits['calibrate'], settings,
    {'base': base_note, 'class_counts': counts, 'dealbreaker_counts': deal_counts, 'health': health,
     'history': result.history, 'best_epoch': result.epoch})
print(lt.questions_report_text(report))


**Before you export:** read the report printed above. A line starting with `!!` means: do not switch to this model; send the report to Claude Code first. To compare it with what you run now, export anyway and run a benchmark of your own on jobs you labelled (`lt.alert_summary` and `lt.go_no_go` do the arithmetic, with your own baseline).


In [ ]:
OUT = Path('/content/drive/MyDrive/laya_questions/laya_questions_v1')
lt.export_questions_checkpoint(result.state, base, base_cfg, OUT, settings, temps, cutoffs, report)
(OUT / 'report.txt').write_text(lt.questions_report_text(report))

files_in_out = sorted(p for p in OUT.rglob('*') if p.is_file())      # before SHA256SUMS exists: it never hashes itself
with (OUT / 'SHA256SUMS').open('w') as sums:
    for p in files_in_out:
        rel = p.relative_to(OUT)
        print(rel, p.stat().st_size, 'bytes')
        sums.write(f'{hashlib.sha256(p.read_bytes()).hexdigest()}  {rel}\n')

zip_path = OUT.parent / f'{OUT.name}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_STORED, allowZip64=True) as zf:
    for p in sorted(OUT.rglob('*')):
        if p.is_file():
            zf.write(p, arcname=str(Path(OUT.name) / p.relative_to(OUT)))

print('Saved to', OUT, '- zipped to', zip_path)
print(f'Download the single {zip_path.name} file (not the folder), unzip it, then inside the unzipped folder run:')
print('  shasum -a 256 -c SHA256SUMS')
print('Every line must say OK.')
print('Then point laya.model in jobhunter.yaml at the unzipped folder, and run a benchmark of your own on labelled '
      'jobs before relying on it.')

try:
    os.sync()
    drive.flush_and_unmount()
except Exception as e:
    print('Could not flush/unmount Drive cleanly:', type(e).__name__,
         '- the files above were already written; safe to ignore if you are about to disconnect anyway.')
